# R-Squared Evaluation for the Selected GAMM

Estimate fixed-effect and conditional R-squared values for the selected model with a participant-level random effect.

## Data and reproducibility

The participant-level NIPT dataset is not distributed in this public repository. To reproduce the analysis, place an authorized copy at `data/nipt_data.xlsx` using the English schema documented in `data/README.md`.


## Setup


In [ ]:
from pathlib import Path

DATA_PATH = Path("data") / "nipt_data.xlsx"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "The authorized NIPT dataset is not included in this public repository. "
        "Place it at data/nipt_data.xlsx after reviewing data-use restrictions."
    )


## Fit the selected model and calculate R-squared


In [ ]:
# -*- coding: utf-8 -*-
# Compute R-squared for m3 with and without the participant random effect
FILE_PATH = DATA_PATH
SHEET_NAME = "male_data"

import sys, subprocess, warnings, re
warnings.filterwarnings("ignore")

def ensure(pkgs):
    for p in pkgs:
        try: __import__(p)
        except Exception:
            subprocess.check_call([sys.executable, "-m", "pip", "install", p, "-q"])
ensure(["pandas","numpy","openpyxl","pygam"])

import numpy as np, pandas as pd
from pygam import LinearGAM, s, te, f


df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, engine="openpyxl")

def parse_weeks(x):
    if pd.isna(x): return np.nan
    if isinstance(x,(int,float)): return float(x)
    s = str(x).strip()
    nums = re.findall(r"\d+(?:\.\d+)?", s)
    if not nums: return np.nan
    if "+" in s or "d" in s or len(nums) >= 2:
        w = float(nums[0]); d = float(nums[1]) if len(nums)>=2 else 0.0
        if d <= 1 and ("+" not in s and "d" not in s):
            return w + d
        return w + d/7.0
    return float(nums[0])

dat = pd.DataFrame({
    "gestw": df["gestational_age"].apply(parse_weeks),
    "bmi":   pd.to_numeric(df["maternal_bmi"], errors="coerce"),
    "id_code": df["maternal_id"].astype("category").cat.codes,
    "y_raw": pd.to_numeric(df["y_chromosome_fraction"], errors="coerce")
})


if dat["y_raw"].max(skipna=True) > 1:
    dat["y_raw"] = dat["y_raw"]/100.0
n_all = dat["y_raw"].notna().sum()
dat["y"] = (dat["y_raw"]*(n_all-1) + 0.5)/n_all


mask = (
    dat["gestw"].between(5,35) &
    dat["bmi"].between(12,45) &
    dat["y"].between(0,1) &
    (dat["id_code"] >= 0)
)
sub = dat.loc[mask].dropna(subset=["gestw","bmi","y"]).copy()
print(f"Sample size n = {len(sub)}, Number of participants = {sub['id_code'].nunique()}")


X_fe   = sub[["gestw","bmi"]].to_numpy()
X_cond = sub[["gestw","bmi","id_code"]].to_numpy()
y      = sub["y"].to_numpy(float)


lam_grid = np.logspace(-3, 3, 7)

def r2_efron(y_true, y_pred):
    tss = np.sum((y_true - y_true.mean())**2)
    rss = np.sum((y_true - y_pred)**2)
    return float(1 - rss/tss) if tss > 0 else float("nan")


m3_fe = LinearGAM(
    s(0, n_splines=6) + s(1, n_splines=5) + te(0,1, n_splines=[6,5])
).gridsearch(X_fe, y, lam=lam_grid, progress=False)
r2_fe = r2_efron(y, m3_fe.predict(X_fe))


m3_cond = LinearGAM(
    s(0, n_splines=6) + s(1, n_splines=5) + te(0,1, n_splines=[6,5]) + f(2)
).gridsearch(X_cond, y, lam=lam_grid, progress=False)
r2_cond = r2_efron(y, m3_cond.predict(X_cond))

print(f"m3（fixed effects only）R2 = {r2_fe:.4f}")
print(f"m3（including participant ID; conditional R-squared） R2 = {r2_cond:.4f}")
